In [1]:
import os
# Force Hugging Face libraries to stay offline
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_EVALUATE_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

In [2]:
import torch
import faiss
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoModelForCausalLM
from diffusers import DDPMScheduler # Useful for the CMD denoising logic

# Metrics for evaluating reconstruction fidelity
from evaluate import load
bleu = load("bleu")
rouge = load("rouge")

# Defense utilities
from opacus import PrivacyEngine  # For implementing DP training

print(f"Environment: CS 6501 Data Privacy")
print(f"PyTorch Version: {torch.version.__version__}")
print(f"GPU Active: {torch.cuda.is_available()}") # Essential for training CMD
print(f"FAISS Version: {faiss.__version__}") # Essential for your embedding store

/home/pmd4nd/.conda/envs/privacy_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using the latest cached version of the module from /home/pmd4nd/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--bleu/9e0985c1200e367cce45605ce0ecb5ede079894e0f24f54613fca08eeb8aff76 (last modified on Thu Mar  5 12:14:48 2026) since it couldn't be found locally at evaluate-metric--bleu, or remotely on the Hugging Face Hub.
Using the latest cached version of the module from /home/pmd4nd/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--rouge/6e5315f72865c2eaa764c8361360bb938740b9c120a2cf3a7ad218aa0ce452ed (last modified on Thu Mar  5 12:14:52 2026) since it couldn't be found locally at evaluate-metric--rouge, or remotely on the Hugging Face Hub.


Environment: CS 6501 Data Privacy
PyTorch Version: 2.4.1+cu121
GPU Active: True
FAISS Version: 1.13.2


# Phase 1: Build Target Database

### Preprocess Data (English-PII-Masking-43k)

1. **Data Extraction**: The script scans your .jsonl file and extracts only the unmasked_text field. In the context of the PII-Masking-200K dataset, this is the "ground truth" sensitive information (like real names, phone numbers, or addresses) that you are trying to protect.

2. **Tokenization**: Since computers don't understand words, the GTR-T5 Tokenizer breaks the text down into "tokens" (sub-word units).

3. **The 32-Token Constraint**: This is the most important part of our methodology. The code applies a strict truncation rule. **Uniformity** -- Every sample is cut off at exactly 32 tokens. Why? As shown in the Vec2Text and Masked Diffusion papers, 32 tokens is the standard "attack window." By limiting your data this way, you are creating a level playing field to measure exactly how much information "leaks" from a single embedding vector.

4. **Cleaning & Decoding**: After truncating, the script decodes the tokens back into text. This removes special characters (like `[PAD]` or `[CLS]`) and ensures that the "Cleaned Text" you save in your `pii_texts.pkl` file matches exactly what the embedding model will process.

In [8]:
import json
from tqdm.auto import tqdm
from transformers import AutoTokenizer

MODEL_LOCAL_PATH = "./gtr-t5-base-local"
limit = 43000

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_LOCAL_PATH)

def process_pii_data(file_path, limit=limit):
    processed_texts = []

    # Verify dataset exists
    if not os.path.exists(file_path):
        print(f"Error: Could not find dataset at {file_path}")
        return []

    with open(file_path, 'r') as f:
        for i, line in tqdm(enumerate(f), total=limit, desc="Processing PII dataset"):
            if i >= limit:
                break

            try:
                data = json.loads(line)

                raw_text = data["unmasked_text"]

                tokens = tokenizer(
                    raw_text,
                    max_length=32,
                    truncation=True,
                    return_tensors="pt"
                )

                clean_text = tokenizer.decode(
                    tokens["input_ids"][0],
                    skip_special_tokens=True
                )

                processed_texts.append(clean_text)

            except json.JSONDecodeError:
                continue

    return processed_texts


# Dataset path
dataset_path = os.path.expanduser("./english_pii_43k.jsonl")

texts_for_db = process_pii_data(dataset_path)

print(f"Successfully processed {len(texts_for_db)} samples.")
print(f"Sample 1: {texts_for_db[0]}")
print(f"Sample 2: {texts_for_db[1]}")

Processing PII dataset: 100%|██████████| 43000/43000 [00:12<00:00, 3470.56it/s]

Successfully processed 43000 samples.
Sample 1: A student's assessment was found on device bearing IMEI: 06-184755-866851-3. The document falls under the various topics discussed
Sample 2: Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access


In [13]:
import pickle

# Save the processed text list to your home directory
with open("processed_pii_texts.pkl", "wb") as f:
    pickle.dump(texts_for_db, f)

print("Progress saved to processed_pii_texts.pkl")

Progress saved to processed_pii_texts.pkl


## Embedding Generator and FAISS Index

### Embedding Generator
This takes the 43,000 "cleaned" text samples from the preprocessor and converts them into Dense Vectors.

* **GTR-T5 Encoder**: It passes the text through the model uploaded. The model looks at the context of the words and outputs a mathematical "summary."

* **Mean Pooling**: The model originally outputs a vector for every single token. "Mean pooling" averages those 32 token-vectors into one single 768-dimensional vector that represents the entire sentence. In the research papers, while mean pooling is "accurate" for retrieval, it is precisely this "summary" that makes the text vulnerable to inversion.

### FAISS Index
Facebook AI Similarity Search (FAISS) is a "Vector Database" developed by Meta.

* **IndexFlatL2**: This specific index stores the 768-dimensional vectors exactly as they are. It doesn't compress them, which is ideal for a privacy audit because we want to test the raw risk of the vectors without any distortion.

* **The "Target"**: We now have a searchable map of the PII. In Phase 2, we will take one of these "mystery vectors" from the index and try to turn it back into the original PII string.

In [5]:
import pickle

# Run this if you already have the processed file
if os.path.exists("processed_pii_texts.pkl"):
    with open("processed_pii_texts.pkl", "rb") as f:
        texts_for_db = pickle.load(f)
    print(f"Ready! Loaded {len(texts_for_db)} pre-processed PII samples.")

Ready! Loaded 43000 pre-processed PII samples.


In [6]:
from tqdm.auto import tqdm
from transformers import T5EncoderModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load encoder-only model
model = T5EncoderModel.from_pretrained(MODEL_LOCAL_PATH)
model.to(device)
model.eval()

def generate_embeddings(texts, batch_size=32):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

            # Mean pooling over token dimension
            embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

print("Generating embeddings...")
vectors = generate_embeddings(texts_for_db)

dimension = vectors.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(vectors.astype("float32"))

print(f"FAISS Index built with {index.ntotal} vectors of dimension {dimension}.")

Loading weights: 100%|██████████| 99/99 [00:00<00:00, 18622.96it/s]


Generating embeddings...


Embedding batches: 100%|██████████| 1344/1344 [00:26<00:00, 50.26it/s]


FAISS Index built with 43000 vectors of dimension 768.


Generating 43,000 embeddings on a CPU can take a while. We can save the index (and embeddings) to avoid running this every time we open Jupyter.

In [15]:
# Save the FAISS index to your home directory
faiss.write_index(index, "pii_audit_target.index")

print("Target Database saved for Phase 2")

# Save NumPy vector embeddings
np.save("embeddings.npy", vectors)

print("NumPy vector embeddings saved")

Target Database saved for Phase 2


This concludes Phase 1. We have files:
* `processed_pii_text.pkl`
* `pii_audit_target.index`